
# Exploratory Analysis of UN General Debate Speeches

## 1. Study Overview and Data Loading

### Research scope

This notebook examines the descriptive characteristics of United Nations General Debate speeches delivered by Iran, the United States, and China between 1946 and 2025.

### Analytical objectives

1. Evaluate country-specific speech length distributions.
2. Examine temporal variation in document length.
3. Compare lexical characteristics across countries.
4. Investigate potential historical and measurement-related differences.
5. Prepare descriptive evidence for subsequent NLP and statistical modeling.

### Methodological principles

- The unit of observation is a country-year speech.
- Missing country-year observations are not imputed.
- Descriptive differences are not interpreted as statistically significant without appropriate inferential analysis.
- Document length and historical period are treated as potential sources of measurement variation.
- The raw and minimally normalized text representations remain available.
- All figures and tables are exported in separate, explicitly labeled cells.

### Data source

The validated speech-level dataset is loaded from:

`data/processed/speeches_master.parquet`


In [ ]:

# ============================================================
# NOTEBOOK 02: EXPLORATORY DATA ANALYSIS
# STEP 01: LOAD AND VALIDATE THE ANALYTICAL DATASET
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

PROJECT_DIR = Path.home() / "Projects" / "7-masoud"

DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "speeches_master.parquet"
)

assert DATA_PATH.is_file(), "Master dataset not found."

# ------------------------------------------------------------
# 2. Load validated data
# ------------------------------------------------------------

df = pd.read_parquet(DATA_PATH)

COUNTRY_LABELS = {
    "IRN": "Iran",
    "USA": "United States",
    "CHN": "China"
}

df["country_name"] = df["country_code"].map(
    COUNTRY_LABELS
)

df = df.sort_values(
    ["country_code", "year"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 3. Validate data structure
# ------------------------------------------------------------

required_columns = [
    "country_code",
    "country_name",
    "year",
    "session",
    "archive_path",
    "word_count",
    "text_raw",
    "text_clean"
]

assert set(required_columns).issubset(df.columns)

assert len(df) == 233
assert df["archive_path"].is_unique

assert not df.duplicated(
    ["country_code", "year", "session"]
).any()

assert df["text_clean"].notna().all()
assert df["word_count"].gt(0).all()

assert df["year"].between(1946, 2025).all()

expected_counts = {
    "IRN": 74,
    "USA": 80,
    "CHN": 79
}

assert (
    df["country_code"].value_counts().to_dict()
    == expected_counts
)

# ------------------------------------------------------------
# 4. Display dataset overview
# ------------------------------------------------------------

print("=" * 65)
print("EXPLORATORY ANALYSIS - DATA VALIDATION")
print("=" * 65)

print(f"Total observations: {len(df):,}")
print(f"Total variables: {df.shape[1]}")
print(f"Study period: {df.year.min()}-{df.year.max()}")

print("\nSpeech counts by country:")

display(
    df.groupby(["country_code", "country_name"])
    .size()
    .rename("n_speeches")
    .reset_index()
)

print("\nMissing values in core variables:")

display(
    df[required_columns]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

print("\nDataset preview:")

display(
    df[
        [
            "country_code",
            "year",
            "session",
            "word_count",
            "filename"
        ]
    ].head(10)
)

print("\nDATASET LOADING: PASSED")



## 2. Descriptive Analysis of Speech Length

### Objective

This section examines differences in speech length among Iran, the United States, and China over the 1946–2025 period.

### Descriptive measures

The analysis includes:

- Sample size and temporal coverage
- Mean, median, and standard deviation
- Minimum and maximum word counts
- Interquartile range (IQR)
- Selected percentiles
- Skewness and coefficient of variation

### Methodological considerations

Speech length may vary substantially across historical periods and countries.

Country-level summary statistics are descriptive and do not account for temporal autocorrelation, historical changes, or differences in the underlying composition of speeches.

The word counts used here are based on the Unicode regular-expression counting procedure established during data preparation. They should not be interpreted as model-specific NLP token counts.

No observations are excluded or transformed in this section.


In [ ]:

# ============================================================
# STEP 02: COUNTRY-LEVEL SPEECH LENGTH STATISTICS
# ============================================================

from scipy.stats import skew

# ------------------------------------------------------------
# 1. Define a descriptive summary function
# ------------------------------------------------------------

def summarize_speech_length(group):
    """Calculate descriptive statistics for speech length."""

    x = group["word_count"].dropna().astype(float)

    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)

    return pd.Series({
        "n": len(x),
        "mean": x.mean(),
        "median": x.median(),
        "std": x.std(ddof=1),
        "min": x.min(),
        "q1": q1,
        "q3": q3,
        "iqr": q3 - q1,
        "p05": x.quantile(0.05),
        "p95": x.quantile(0.95),
        "max": x.max(),
        "skewness": skew(x, bias=False),
        "cv": x.std(ddof=1) / x.mean()
    })


# ------------------------------------------------------------
# 2. Calculate country-level statistics
# ------------------------------------------------------------

length_summary = (
    df.groupby(["country_code", "country_name"])
    .apply(
        summarize_speech_length,
        include_groups=False
    )
    .reset_index()
)

country_order = ["IRN", "USA", "CHN"]

length_summary["country_code"] = pd.Categorical(
    length_summary["country_code"],
    categories=country_order,
    ordered=True
)

length_summary = (
    length_summary
    .sort_values("country_code")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Validate descriptive results
# ------------------------------------------------------------

assert int(length_summary["n"].sum()) == len(df)

assert length_summary["iqr"].ge(0).all()
assert length_summary["std"].ge(0).all()
assert length_summary["mean"].gt(0).all()

# ------------------------------------------------------------
# 4. Display results
# ------------------------------------------------------------

print("=" * 70)
print("SPEECH LENGTH: COUNTRY-LEVEL DESCRIPTIVE STATISTICS")
print("=" * 70)

display(length_summary.round(2))

print("\nTotal observations:", int(length_summary["n"].sum()))
print("Descriptive statistics: PASSED")


In [ ]:

# ============================================================
# STEP 02.1: SPEECH LENGTH DISTRIBUTION VISUALIZATION
# ============================================================

plot_order = ["Iran", "United States", "China"]

fig_length, ax_length = plt.subplots(
    figsize=(9, 5.5)
)

sns.boxplot(
    data=df,
    x="country_name",
    y="word_count",
    order=plot_order,
    showfliers=False,
    color="lightgray",
    width=0.5,
    ax=ax_length
)

sns.stripplot(
    data=df,
    x="country_name",
    y="word_count",
    order=plot_order,
    color="black",
    alpha=0.4,
    size=3.5,
    jitter=0.2,
    ax=ax_length
)

ax_length.set(
    title="Distribution of UN General Debate Speech Lengths",
    xlabel="Country",
    ylabel="Speech Length (Words)"
)

ax_length.grid(
    axis="y",
    alpha=0.2
)

ax_length.set_axisbelow(True)

fig_length.tight_layout()
plt.show()



### Supplementary Figure S1 — Speech Length Distribution

This supplementary figure presents the distribution of speech lengths across Iran, the United States, and China.

Boxplots display the median and interquartile range, with whiskers extending to observations within 1.5 times the interquartile range. Individual points represent observed speeches.

The visualization is descriptive and does not account for temporal variation or dependence between observations.

The figure is exported separately from the visualization code in publication-compatible vector and raster formats.


In [ ]:

# # ============================================================
# # FIGURE EXPORT: SUPPLEMENTARY FIGURE S1
# # Speech Length Distribution by Country
# # ============================================================

# FIGURE_NAME = "FigS01_Speech_Length_Distribution"

# FIGURE_DIR = PROJECT_DIR / "results"

# assert "fig_length" in globals(), (
#     "Run the speech length visualization cell first."
# )

# # Export vector format
# fig_length.savefig(
#     FIGURE_DIR / f"{FIGURE_NAME}.pdf",
#     format="pdf",
#     bbox_inches="tight"
# )

# # Export high-resolution raster format
# fig_length.savefig(
#     FIGURE_DIR / f"{FIGURE_NAME}.png",
#     format="png",
#     dpi=600,
#     bbox_inches="tight"
# )

# print("Supplementary Figure S1 exported successfully.")
# print(f"Location: {FIGURE_DIR}")



### Supplementary Table S1 — Descriptive Speech Length Statistics

This table reports country-specific descriptive statistics for speech lengths, including sample size, central tendency, dispersion, selected percentiles, skewness, and coefficient of variation.

Values are based on all available speeches for each country between 1946 and 2025.

The reported statistics are descriptive and do not constitute inferential comparisons between countries.


In [ ]:

# # ============================================================
# # TABLE EXPORT: SUPPLEMENTARY TABLE S1
# # Country-Level Speech Length Statistics
# # ============================================================

# TABLE_NAME = "TableS01_Speech_Length_Descriptive_Statistics"

# TABLE_DIR = PROJECT_DIR / "results"

# assert "length_summary" in globals(), (
#     "Run the descriptive statistics cell first."
# )

# table_export = length_summary.copy()

# # Convert categorical identifiers to ordinary strings.
# table_export["country_code"] = (
#     table_export["country_code"].astype(str)
# )

# # Preserve numerical precision in the source table.
# table_export.to_excel(
#     TABLE_DIR / f"{TABLE_NAME}.xlsx",
#     index=False,
#     engine="openpyxl"
# )

# table_export.to_csv(
#     TABLE_DIR / f"{TABLE_NAME}.csv",
#     index=False,
#     encoding="utf-8"
# )

# print("Supplementary Table S1 exported successfully.")
# print(f"Location: {TABLE_DIR}")



## 3. Temporal Dynamics of Speech Length

### Objective

This section examines the temporal evolution of speech length among Iran, the United States, and China between 1946 and 2025.

### Analytical strategy

1. Examine annual speech lengths without temporal aggregation.
2. Calculate descriptive rolling averages to visualize long-term patterns.
3. Identify years with unusually long or short speeches.
4. Compare speech lengths across countries during overlapping years.
5. Quantify descriptive within-year differences in speech length.

### Methodological considerations

The corpus contains one speech per country-year observation, with some missing years.

Rolling averages must not treat missing country-year observations as zero-length speeches. A five-year calendar window is therefore used with a complete annual index.

Rolling averages are descriptive smoothing tools, not statistical estimates of structural changes.

Differences in speech length must not be interpreted as differences in diplomatic influence, political importance, or rhetorical intensity.

Formal inference, autocorrelation diagnostics, uncertainty estimation, and structural-change analysis will be conducted separately.


In [ ]:

# ============================================================
# STEP 03: TEMPORAL SPEECH LENGTH DATA PREPARATION
# ============================================================

YEAR_START = 1946
YEAR_END = 2025
ROLLING_WINDOW = 5

country_order = ["IRN", "USA", "CHN"]

# ------------------------------------------------------------
# 1. Construct a complete country-year calendar
# ------------------------------------------------------------

calendar_index = pd.MultiIndex.from_product(
    [
        country_order,
        range(YEAR_START, YEAR_END + 1)
    ],
    names=["country_code", "year"]
)

temporal_data = (
    df.set_index(["country_code", "year"])
    .reindex(calendar_index)
    .reset_index()
)

temporal_data["country_name"] = (
    temporal_data["country_code"]
    .map(COUNTRY_LABELS)
)

# ------------------------------------------------------------
# 2. Preserve missing observations explicitly
# ------------------------------------------------------------

temporal_data["speech_available"] = (
    temporal_data["word_count"].notna()
)

assert len(temporal_data) == 3 * 80

assert (
    temporal_data["speech_available"].sum() == 233
)

assert (
    temporal_data.loc[
        ~temporal_data["speech_available"],
        "word_count"
    ].isna().all()
)

# ------------------------------------------------------------
# 3. Calculate five-year rolling averages
# ------------------------------------------------------------

# Centered calendar-year window with minimum 3 observations.
temporal_data = temporal_data.sort_values(
    ["country_code", "year"]
).reset_index(drop=True)

temporal_data["rolling_mean_5y"] = (
    temporal_data
    .groupby("country_code")["word_count"]
    .transform(
        lambda s: s.rolling(
            window=ROLLING_WINDOW,
            center=True,
            min_periods=3
        ).mean()
    )
)

temporal_data["rolling_n_5y"] = (
    temporal_data
    .groupby("country_code")["word_count"]
    .transform(
        lambda s: s.rolling(
            window=ROLLING_WINDOW,
            center=True,
            min_periods=1
        ).count()
    )
)

# ------------------------------------------------------------
# 4. Validation checks
# ------------------------------------------------------------

assert (
    temporal_data.loc[
        temporal_data["rolling_n_5y"] < 3,
        "rolling_mean_5y"
    ].isna().all()
)

assert (
    temporal_data["rolling_n_5y"] <= 5
).all()

# ------------------------------------------------------------
# 5. Report temporal data quality
# ------------------------------------------------------------

print("=" * 70)
print("TEMPORAL SPEECH LENGTH DATA PREPARATION")
print("=" * 70)

print("Calendar observations:", len(temporal_data))
print(
    "Observed speeches:",
    int(temporal_data["speech_available"].sum())
)

print(
    "Missing country-year observations:",
    int((~temporal_data["speech_available"]).sum())
)

print("\nMissing years by country:")

display(
    temporal_data.groupby("country_code")
    ["speech_available"]
    .agg(
        observed="sum",
        total="count"
    )
    .assign(
        missing=lambda x: x["total"] - x["observed"]
    )
)

print("\nRolling-average availability:")

display(
    temporal_data.groupby("country_code")
    ["rolling_mean_5y"]
    .agg(
        available="count",
        missing=lambda x: x.isna().sum()
    )
)

print("\nTemporal data validation: PASSED")


In [ ]:

# ============================================================
# STEP 03.3: REVISED TEMPORAL SPEECH LENGTH FIGURE
# ============================================================

fig_temporal, axes = plt.subplots(
    3, 1,
    figsize=(12, 9),
    sharex=True,
    sharey=True
)

for ax, code in zip(axes, country_order):

    subset = temporal_data.loc[
        temporal_data["country_code"] == code
    ].sort_values("year").copy()

    observed = subset["speech_available"]

    # Display the rolling mean only at observed years.
    subset["display_rolling_mean"] = (
        subset["rolling_mean_5y"].where(observed)
    )

    ax.scatter(
        subset.loc[observed, "year"],
        subset.loc[observed, "word_count"],
        color="gray",
        alpha=0.45,
        s=20,
        label="Observed speech"
    )

    ax.plot(
        subset["year"],
        subset["display_rolling_mean"],
        color="black",
        linewidth=2,
        label="Centered 5-year rolling mean"
    )

    ax.set_title(
        COUNTRY_LABELS[code],
        loc="left",
        fontweight="bold",
        fontsize=12
    )

    ax.set_ylabel("Word count")
    ax.grid(alpha=0.2)
    ax.set_axisbelow(True)

axes[-1].set_xlabel("Year")
axes[-1].set_xlim(1945, 2026)

axes[0].legend(
    frameon=False,
    loc="upper right"
)

fig_temporal.suptitle(
    "Historical Variation in UN General Debate Speech Length",
    fontsize=14
)

fig_temporal.tight_layout()
plt.show()



### 3.2. Matched-Year Comparisons of Speech Length

#### Objective

Compare speech lengths between countries within the same calendar years to reduce differences caused by unequal temporal coverage.

#### Pairwise comparisons

- Iran versus the United States
- China versus the United States
- Iran versus China

#### Measures

For each country pair, the analysis calculates:

- Number of overlapping observed years
- Mean and median within-year word-count differences
- Standard deviation of paired differences
- Minimum and maximum paired differences
- Proportion of overlapping years with a positive difference

#### Interpretation

Paired comparisons are descriptive and account for calendar-year matching, but do not eliminate temporal dependence or other potential confounding factors.

No statistical significance tests are conducted at this stage.


In [ ]:

# ============================================================
# STEP 03.2: MATCHED-YEAR SPEECH LENGTH COMPARISONS
# ============================================================

# ------------------------------------------------------------
# 1. Create country-year wide-format data
# ------------------------------------------------------------

length_wide = df.pivot(
    index="year",
    columns="country_code",
    values="word_count"
).reindex(
    range(YEAR_START, YEAR_END + 1)
)

# ------------------------------------------------------------
# 2. Define pairwise comparison function
# ------------------------------------------------------------

def paired_length_summary(data, first, second):
    """Summarize within-year differences in speech length."""

    matched = data[[first, second]].dropna()

    differences = (
        matched[first] - matched[second]
    )

    return {
        "comparison": f"{first} - {second}",
        "n_matched_years": len(matched),
        "mean_difference": differences.mean(),
        "median_difference": differences.median(),
        "sd_difference": differences.std(ddof=1),
        "min_difference": differences.min(),
        "max_difference": differences.max(),
        "positive_difference_pct": (
            100 * differences.gt(0).mean()
        )
    }


# ------------------------------------------------------------
# 3. Calculate pairwise summaries
# ------------------------------------------------------------

pairs = [
    ("IRN", "USA"),
    ("CHN", "USA"),
    ("IRN", "CHN")
]

paired_summary = pd.DataFrame([
    paired_length_summary(length_wide, a, b)
    for a, b in pairs
])

# ------------------------------------------------------------
# 4. Validate matched-year counts
# ------------------------------------------------------------

expected_matched_years = {
    "IRN - USA": 74,
    "CHN - USA": 79,
    "IRN - CHN": 73
}

for row in paired_summary.itertuples(index=False):
    assert (
        row.n_matched_years
        == expected_matched_years[row.comparison]
    )

# ------------------------------------------------------------
# 5. Display descriptive comparisons
# ------------------------------------------------------------

print("=" * 70)
print("MATCHED-YEAR SPEECH LENGTH COMPARISONS")
print("=" * 70)

display(paired_summary.round(2))

print("\nPaired comparison validation: PASSED")


### Save figure

In [ ]:

# # ============================================================
# # FIGURE EXPORT: SUPPLEMENTARY FIGURE S02
# # Historical Speech Length Trends
# # ============================================================

# figure_name = "FigS02_Historical_Speech_Length_Trends"

# figure_dir = PROJECT_DIR / "results"

# assert fig_temporal is not None
# assert len(fig_temporal.axes) == 3

# pdf_path = figure_dir / f"{figure_name}.pdf"
# png_path = figure_dir / f"{figure_name}.png"

# fig_temporal.savefig(
#     pdf_path,
#     bbox_inches="tight",
#     format="pdf"
# )

# fig_temporal.savefig(
#     png_path,
#     bbox_inches="tight",
#     dpi=600,
#     format="png"
# )

# print("Supplementary Figure S02 exported.")
# print(pdf_path)
# print(png_path)


### Save Table

In [ ]:

# # ============================================================
# # TABLE EXPORT: SUPPLEMENTARY TABLE S02
# # Matched-Year Speech Length Comparisons
# # ============================================================

# table_name = "TableS02_Matched_Year_Speech_Length"

# table_dir = PROJECT_DIR / "results"

# table_export = paired_summary.copy()

# table_export.to_excel(
#     table_dir / f"{table_name}.xlsx",
#     index=False
# )

# table_export.to_csv(
#     table_dir / f"{table_name}.csv",
#     index=False,
#     encoding="utf-8"
# )

# print("Supplementary Table S02 exported.")



## 4. Decade-Level Coverage and Speech Length Diagnostics

### Objective

Assess temporal heterogeneity in speech availability and document length across the three-country analytical cohort.

### Analytical approach

- Group observations into calendar decades.
- Report the number of observed speeches and expected country-year observations.
- Calculate decade-specific speech length statistics.
- Identify periods with incomplete country coverage.
- Evaluate whether pooled country comparisons may be influenced by historical variation in speech length.

### Interpretation

Calendar decades are used for descriptive diagnostics only and do not represent predefined geopolitical regimes.

The final geopolitical periodization will be specified separately using historically documented events and theoretical considerations.

No missing speech lengths are imputed, and no inferential tests are conducted.


In [ ]:

# ============================================================
# STEP 04: DECADE-LEVEL SPEECH LENGTH DIAGNOSTICS
# ============================================================

# ------------------------------------------------------------
# 1. Assign calendar decades
# ------------------------------------------------------------

decade_data = temporal_data.copy()

decade_data["decade"] = (
    decade_data["year"] // 10 * 10
)

# ------------------------------------------------------------
# 2. Calculate decade-level statistics
# ------------------------------------------------------------

decade_summary = (
    decade_data
    .groupby(
        ["country_code", "decade"],
        observed=True
    )
    .agg(
        expected_years=("year", "size"),
        observed_speeches=("speech_available", "sum"),
        mean_words=("word_count", "mean"),
        median_words=("word_count", "median"),
        sd_words=("word_count", "std"),
        min_words=("word_count", "min"),
        max_words=("word_count", "max")
    )
    .reset_index()
)

decade_summary["missing_years"] = (
    decade_summary["expected_years"]
    - decade_summary["observed_speeches"]
)

decade_summary["coverage_pct"] = (
    100
    * decade_summary["observed_speeches"]
    / decade_summary["expected_years"]
)

# ------------------------------------------------------------
# 3. Validate temporal accounting
# ------------------------------------------------------------

assert decade_summary["expected_years"].sum() == 240
assert decade_summary["observed_speeches"].sum() == 233
assert decade_summary["missing_years"].sum() == 7

assert decade_summary["coverage_pct"].between(
    0, 100
).all()

# ------------------------------------------------------------
# 4. Display results
# ------------------------------------------------------------

print("=" * 70)
print("DECADE-LEVEL SPEECH LENGTH DIAGNOSTICS")
print("=" * 70)

display(
    decade_summary.sort_values(
        ["country_code", "decade"]
    ).round(2)
)

print("\nDecade-level coverage matrix:")

coverage_matrix = decade_summary.pivot(
    index="decade",
    columns="country_code",
    values="coverage_pct"
)

display(coverage_matrix.round(1))

print("\nDecade-level median word counts:")

median_matrix = decade_summary.pivot(
    index="decade",
    columns="country_code",
    values="median_words"
)

display(median_matrix.round(1))

print("\nDecade-level diagnostics: PASSED")



## 5. Lexical Characteristics and Document Comparability

### Objective

Evaluate basic lexical characteristics of the three-country speech corpus before applying advanced natural language processing methods.

### Measures

1. **Lexical token count:** Number of tokens identified using a consistent Unicode-aware lexical pattern.
2. **Vocabulary size:** Number of unique lowercased lexical tokens in each speech.
3. **Type–Token Ratio (TTR):** Vocabulary size divided by lexical token count.
4. **Mean Segmental Type–Token Ratio (MSTTR):** Mean TTR across consecutive, non-overlapping segments of 100 lexical tokens.
5. **Segmentation coverage:** Proportion of lexical tokens represented in complete segments.

### Methodological considerations

TTR is sensitive to document length and should not be used as the sole basis for comparing lexical diversity.

MSTTR reduces some length-related bias but remains sensitive to tokenization, segment size, and text composition.

A common tokenizer is applied to all documents. Lowercasing is used only for calculating lexical diversity; the original text representations remain unchanged.

The final incomplete segment is excluded from MSTTR calculation, and the corresponding token coverage is reported.

These measures describe lexical characteristics rather than diplomatic sentiment, foreign-policy priorities, or rhetorical distance.

No formal significance testing is performed in this section.


In [ ]:

# ============================================================
# STEP 05: LEXICAL CHARACTERISTICS
# ============================================================

import re
import numpy as np
import pandas as pd

SEGMENT_SIZE = 100

# Unicode-aware alphabetic words with optional apostrophes.
LEXICAL_PATTERN = re.compile(
    r"[^\W\d_]+(?:['\u2019][^\W\d_]+)*",
    flags=re.UNICODE
)

# ------------------------------------------------------------
# 1. Define lexical measurement functions
# ------------------------------------------------------------

def extract_lexical_tokens(text):
    """Extract lexical tokens with consistent case folding."""
    return [
        match.group().casefold()
        for match in LEXICAL_PATTERN.finditer(text)
    ]


def calculate_msttr(tokens, segment_size=100):
    """Calculate mean TTR across complete segments only."""

    n_segments = len(tokens) // segment_size

    if n_segments == 0:
        return np.nan, 0, 0.0

    segment_ratios = []

    for segment_index in range(n_segments):
        start = segment_index * segment_size
        end = start + segment_size

        segment = tokens[start:end]

        segment_ratios.append(
            len(set(segment)) / segment_size
        )

    coverage = (
        n_segments * segment_size / len(tokens)
    )

    return (
        float(np.mean(segment_ratios)),
        n_segments,
        coverage
    )


def calculate_lexical_features(text):
    """Calculate document-level lexical characteristics."""

    tokens = extract_lexical_tokens(text)
    token_count = len(tokens)
    vocabulary_size = len(set(tokens))

    msttr, n_segments, coverage = calculate_msttr(
        tokens,
        segment_size=SEGMENT_SIZE
    )

    return {
        "lexical_token_count": token_count,
        "vocabulary_size": vocabulary_size,
        "ttr": (
            vocabulary_size / token_count
            if token_count else np.nan
        ),
        "msttr_100": msttr,
        "msttr_segments": n_segments,
        "msttr_coverage": coverage
    }


# ------------------------------------------------------------
# 2. Validate measurement functions
# ------------------------------------------------------------

assert extract_lexical_tokens(
    "Peace, security, and development."
) == ["peace", "security", "and", "development"]

test_tokens = ["peace", "security"] * 100

test_msttr, test_n, test_coverage = calculate_msttr(
    test_tokens,
    segment_size=100
)

assert np.isclose(test_msttr, 0.02)
assert test_n == 2
assert np.isclose(test_coverage, 1.0)

print("Lexical function tests: PASSED")

# ------------------------------------------------------------
# 3. Compute features for the full analytical cohort
# ------------------------------------------------------------

lexical_records = [
    calculate_lexical_features(text)
    for text in df["text_clean"]
]

lexical_features = pd.DataFrame(
    lexical_records,
    index=df.index
)

lexical_df = pd.concat(
    [
        df[
            ["country_code", "country_name",
             "year", "archive_path", "word_count"]
        ].copy(),
        lexical_features
    ],
    axis=1
)

# ------------------------------------------------------------
# 4. Validate feature extraction
# ------------------------------------------------------------

assert len(lexical_df) == len(df) == 233
assert lexical_df["archive_path"].is_unique
assert lexical_df["lexical_token_count"].gt(0).all()
assert lexical_df["vocabulary_size"].gt(0).all()

assert lexical_df["ttr"].between(0, 1).all()
assert lexical_df["msttr_100"].between(0, 1).all()
assert lexical_df["msttr_coverage"].between(0, 1).all()
assert lexical_df["msttr_segments"].ge(1).all()

# ------------------------------------------------------------
# 5. Summarize country-level lexical characteristics
# ------------------------------------------------------------

lexical_summary = (
    lexical_df
    .groupby(["country_code", "country_name"])
    .agg(
        n_speeches=("archive_path", "size"),
        median_tokens=("lexical_token_count", "median"),
        median_vocabulary=("vocabulary_size", "median"),
        mean_ttr=("ttr", "mean"),
        median_ttr=("ttr", "median"),
        mean_msttr=("msttr_100", "mean"),
        median_msttr=("msttr_100", "median"),
        min_segment_coverage=("msttr_coverage", "min")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------

print("=" * 70)
print("LEXICAL CHARACTERISTICS: THREE-COUNTRY COMPARISON")
print("=" * 70)

display(lexical_summary.round(4))

print("\nLexical token counts versus previous word counts:")

display(
    lexical_df.assign(
        token_count_difference=lambda x:
            x["lexical_token_count"] - x["word_count"]
    )[
        ["country_code", "year",
         "word_count", "lexical_token_count",
         "token_count_difference"]
    ].sort_values(
        "token_count_difference",
        key=lambda x: x.abs(),
        ascending=False
    ).head(10)
)

print("\nLexical feature extraction: PASSED")



## 6. Robustness of Lexical Diversity Measurements

### Objective

Evaluate the sensitivity of lexical diversity estimates to document length and the choice of segment size.

### Analytical strategy

1. Recalculate MSTTR using segment sizes of 50, 100, and 200 tokens.
2. Compare country-level mean and median estimates.
3. Calculate Spearman correlations between document length and lexical diversity indices.
4. Examine whether descriptive country rankings change across segment sizes.

### Methodological considerations

MSTTR is not entirely independent of document length or segmentation choices.

Spearman correlations are used as descriptive diagnostics. Their values are not interpreted as evidence of statistical significance because observations may exhibit temporal dependence.

Country rankings are descriptive summaries rather than measures of national linguistic ability or diplomatic performance.

No statistical hypothesis tests, document exclusions, or source-data modifications are performed.


In [ ]:

# ============================================================
# STEP 06: LEXICAL DIVERSITY ROBUSTNESS ASSESSMENT
# ============================================================

from scipy.stats import spearmanr

SEGMENT_SIZES = [50, 100, 200]

# ------------------------------------------------------------
# 1. Recalculate MSTTR at alternative segment sizes
# ------------------------------------------------------------

robustness_df = lexical_df.copy()

tokenized_documents = [
    extract_lexical_tokens(text)
    for text in df["text_clean"]
]

for segment_size in SEGMENT_SIZES:

    values = [
        calculate_msttr(tokens, segment_size)
        for tokens in tokenized_documents
    ]

    robustness_df[f"msttr_{segment_size}"] = [
        result[0] for result in values
    ]

    robustness_df[f"coverage_{segment_size}"] = [
        result[2] for result in values
    ]

# ------------------------------------------------------------
# 2. Validate calculations
# ------------------------------------------------------------

assert len(robustness_df) == 233

assert np.allclose(
    robustness_df["msttr_100"],
    lexical_df["msttr_100"],
    equal_nan=True
)

for segment_size in SEGMENT_SIZES:
    column = f"msttr_{segment_size}"

    assert robustness_df[column].notna().all()
    assert robustness_df[column].between(0, 1).all()

# ------------------------------------------------------------
# 3. Country-level sensitivity summary
# ------------------------------------------------------------

msttr_columns = [
    f"msttr_{size}" for size in SEGMENT_SIZES
]

msttr_summary = (
    robustness_df
    .groupby("country_code")[msttr_columns]
    .agg(["mean", "median", "std"])
)

print("=" * 70)
print("MSTTR SEGMENT-SIZE SENSITIVITY")
print("=" * 70)

display(msttr_summary.round(4))

# ------------------------------------------------------------
# 4. Examine document-length associations
# ------------------------------------------------------------

correlation_records = []

for country, subset in robustness_df.groupby("country_code"):

    for measure in ["ttr"] + msttr_columns:

        rho, _ = spearmanr(
            subset["lexical_token_count"],
            subset[measure]
        )

        correlation_records.append({
            "country_code": country,
            "measure": measure,
            "n_speeches": len(subset),
            "spearman_rho": rho
        })

length_correlations = pd.DataFrame(
    correlation_records
)

print("\nDocument length versus lexical diversity:")

display(
    length_correlations.pivot(
        index="measure",
        columns="country_code",
        values="spearman_rho"
    ).round(3)
)

# ------------------------------------------------------------
# 5. Compare descriptive country rankings
# ------------------------------------------------------------

rank_records = []

for measure in ["ttr"] + msttr_columns:

    country_means = (
        robustness_df.groupby("country_code")[measure]
        .mean()
        .sort_values(ascending=False)
    )

    rank_records.append({
        "measure": measure,
        "descending_country_order": " > ".join(
            country_means.index.tolist()
        )
    })

ranking_summary = pd.DataFrame(rank_records)

print("\nSensitivity of descriptive country ordering:")
display(ranking_summary)

print("\nLexical robustness diagnostics: PASSED")

# ============================================================
# STEP 06: LEXICAL DIVERSITY ROBUSTNESS ASSESSMENT
# ============================================================

from scipy.stats import spearmanr

SEGMENT_SIZES = [50, 100, 200]

# ------------------------------------------------------------
# 1. Recalculate MSTTR at alternative segment sizes
# ------------------------------------------------------------

robustness_df = lexical_df.copy()

tokenized_documents = [
    extract_lexical_tokens(text)
    for text in df["text_clean"]
]

for segment_size in SEGMENT_SIZES:

    values = [
        calculate_msttr(tokens, segment_size)
        for tokens in tokenized_documents
    ]

    robustness_df[f"msttr_{segment_size}"] = [
        result[0] for result in values
    ]

    robustness_df[f"coverage_{segment_size}"] = [
        result[2] for result in values
    ]

# ------------------------------------------------------------
# 2. Validate calculations
# ------------------------------------------------------------

assert len(robustness_df) == 233

assert np.allclose(
    robustness_df["msttr_100"],
    lexical_df["msttr_100"],
    equal_nan=True
)

for segment_size in SEGMENT_SIZES:
    column = f"msttr_{segment_size}"

    assert robustness_df[column].notna().all()
    assert robustness_df[column].between(0, 1).all()

# ------------------------------------------------------------
# 3. Country-level sensitivity summary
# ------------------------------------------------------------

msttr_columns = [
    f"msttr_{size}" for size in SEGMENT_SIZES
]

msttr_summary = (
    robustness_df
    .groupby("country_code")[msttr_columns]
    .agg(["mean", "median", "std"])
)

print("=" * 70)
print("MSTTR SEGMENT-SIZE SENSITIVITY")
print("=" * 70)

display(msttr_summary.round(4))

# ------------------------------------------------------------
# 4. Examine document-length associations
# ------------------------------------------------------------

correlation_records = []

for country, subset in robustness_df.groupby("country_code"):

    for measure in ["ttr"] + msttr_columns:

        rho, _ = spearmanr(
            subset["lexical_token_count"],
            subset[measure]
        )

        correlation_records.append({
            "country_code": country,
            "measure": measure,
            "n_speeches": len(subset),
            "spearman_rho": rho
        })

length_correlations = pd.DataFrame(
    correlation_records
)

print("\nDocument length versus lexical diversity:")

display(
    length_correlations.pivot(
        index="measure",
        columns="country_code",
        values="spearman_rho"
    ).round(3)
)

# ------------------------------------------------------------
# 5. Compare descriptive country rankings
# ------------------------------------------------------------

rank_records = []

for measure in ["ttr"] + msttr_columns:

    country_means = (
        robustness_df.groupby("country_code")[measure]
        .mean()
        .sort_values(ascending=False)
    )

    rank_records.append({
        "measure": measure,
        "descending_country_order": " > ".join(
            country_means.index.tolist()
        )
    })

ranking_summary = pd.DataFrame(rank_records)

print("\nSensitivity of descriptive country ordering:")
display(ranking_summary)

print("\nLexical robustness diagnostics: PASSED")



### Supplementary Table S03 — Lexical Diversity Sensitivity

This supplementary table documents the sensitivity of lexical diversity measurements to MSTTR segment size.

The table includes country-level estimates for segment sizes of 50, 100, and 200 tokens, as well as descriptive Spearman correlations between document length and lexical diversity.

Results are descriptive and are not adjusted for longitudinal dependence or historical differences in text production.

The supplementary table is exported independently from the analytical computations.


In [ ]:

# # ============================================================
# # TABLE EXPORT: SUPPLEMENTARY TABLE S03
# # Lexical Diversity Sensitivity Analysis
# # ============================================================

# from pathlib import Path

# table_path = (
#     PROJECT_DIR
#     / "results"
#     / "TableS03_Lexical_Diversity_Sensitivity.xlsx"
# )

# with pd.ExcelWriter(
#     table_path,
#     engine="openpyxl"
# ) as writer:

#     msttr_summary.to_excel(
#         writer,
#         sheet_name="MSTTR_Sensitivity"
#     )

#     length_correlations.to_excel(
#         writer,
#         sheet_name="Length_Correlations",
#         index=False
#     )

#     ranking_summary.to_excel(
#         writer,
#         sheet_name="Descriptive_Ordering",
#         index=False
#     )

# print("Supplementary Table S03 exported.")
# print(table_path)



## 7. Historical Evolution of Lexical Diversity

### Objective

Examine the temporal evolution of lexical diversity among Iran, the United States, and China using MSTTR calculated over 100-token segments.

### Analytical strategy

1. Construct chronological lexical diversity series for each country.
2. Preserve missing country-year observations.
3. Calculate centered five-year rolling means using calendar-year windows.
4. Visualize annual measurements and smoothed descriptive patterns.
5. Identify potential differences across historical periods for subsequent statistical investigation.

### Methodological limitations

Rolling averages describe local variation and do not establish statistically significant trends or structural breaks.

MSTTR remains sensitive to segmentation choices, translation practices, linguistic conventions, and document composition.

The historical representation of China and changes in UNGDC text-production methods require consideration before substantive interpretation.

No imputation, formal hypothesis testing, or document exclusion is performed.


In [ ]:

# ============================================================
# STEP 07: TEMPORAL LEXICAL DIVERSITY PREPARATION
# ============================================================

# ------------------------------------------------------------
# 1. Construct complete calendar-year observations
# ------------------------------------------------------------

lexical_calendar = pd.MultiIndex.from_product(
    [
        ["IRN", "USA", "CHN"],
        range(1946, 2026)
    ],
    names=["country_code", "year"]
)

lexical_temporal = (
    robustness_df
    .set_index(["country_code", "year"])
    .reindex(lexical_calendar)
    .reset_index()
)

lexical_temporal["country_name"] = (
    lexical_temporal["country_code"].map(COUNTRY_LABELS)
)

# ------------------------------------------------------------
# 2. Calculate centered five-year rolling statistics
# ------------------------------------------------------------

lexical_temporal = lexical_temporal.sort_values(
    ["country_code", "year"]
).reset_index(drop=True)

lexical_temporal["msttr_rolling_5y"] = (
    lexical_temporal
    .groupby("country_code")["msttr_100"]
    .transform(
        lambda s: s.rolling(
            window=5,
            center=True,
            min_periods=3
        ).mean()
    )
)

lexical_temporal["rolling_n_5y"] = (
    lexical_temporal
    .groupby("country_code")["msttr_100"]
    .transform(
        lambda s: s.rolling(
            window=5,
            center=True,
            min_periods=1
        ).count()
    )
)

# ------------------------------------------------------------
# 3. Validate temporal structure
# ------------------------------------------------------------

assert len(lexical_temporal) == 240
assert lexical_temporal["msttr_100"].notna().sum() == 233

assert (
    lexical_temporal.loc[
        lexical_temporal["rolling_n_5y"] < 3,
        "msttr_rolling_5y"
    ].isna().all()
)

assert lexical_temporal["msttr_rolling_5y"].dropna().between(
    0, 1
).all()

# ------------------------------------------------------------
# 4. Report results
# ------------------------------------------------------------

print("=" * 70)
print("TEMPORAL LEXICAL DIVERSITY VALIDATION")
print("=" * 70)

print("Calendar observations:", len(lexical_temporal))
print(
    "Observed speeches:",
    lexical_temporal["msttr_100"].notna().sum()
)

print("\nRolling-average availability:")

display(
    lexical_temporal.groupby("country_code")
    .agg(
        observed_years=("msttr_100", "count"),
        rolling_estimates=("msttr_rolling_5y", "count")
    )
)

print("\nTemporal lexical validation: PASSED")


In [ ]:

# ============================================================
# STEP 07.1: TEMPORAL LEXICAL DIVERSITY VISUALIZATION
# ============================================================

fig_lexical_trends, axes = plt.subplots(
    3, 1,
    figsize=(12, 9),
    sharex=True,
    sharey=True
)

for ax, code in zip(
    axes,
    ["IRN", "USA", "CHN"]
):

    subset = lexical_temporal.loc[
        lexical_temporal["country_code"] == code
    ].sort_values("year").copy()

    observed = subset["msttr_100"].notna()

    # Prevent lines from spanning missing annual observations.
    subset["display_rolling_mean"] = (
        subset["msttr_rolling_5y"].where(observed)
    )

    ax.scatter(
        subset.loc[observed, "year"],
        subset.loc[observed, "msttr_100"],
        color="gray",
        alpha=0.55,
        s=22,
        label="Annual MSTTR"
    )

    ax.plot(
        subset["year"],
        subset["display_rolling_mean"],
        color="black",
        linewidth=2,
        label="Centered 5-year rolling mean"
    )

    ax.set_title(
        COUNTRY_LABELS[code],
        loc="left",
        fontweight="bold"
    )

    ax.set_ylabel("MSTTR-100")
    ax.grid(alpha=0.2)
    ax.set_axisbelow(True)

axes[-1].set_xlabel("Year")
axes[-1].set_xlim(1945, 2026)

axes[0].legend(
    frameon=False,
    loc="upper right"
)

fig_lexical_trends.suptitle(
    "Historical Variation in Lexical Diversity",
    fontsize=14
)

fig_lexical_trends.tight_layout()
plt.show()



### Supplementary Figure S03 — Historical Lexical Diversity

This figure illustrates annual MSTTR-100 values and centered five-year rolling averages for Iran, the United States, and China.

Rolling averages use a calendar-based window requiring at least three observed speeches. Lines are interrupted at missing speech years.

The figure provides descriptive evidence only. It does not establish statistically significant trends or changes in diplomatic discourse.


In [ ]:

# # ============================================================
# # FIGURE EXPORT: SUPPLEMENTARY FIGURE S03
# # Historical Lexical Diversity
# # ============================================================

# figure_name = "FigS03_Historical_Lexical_Diversity"
# figure_dir = PROJECT_DIR / "results"

# assert fig_lexical_trends is not None
# assert len(fig_lexical_trends.axes) == 3

# pdf_path = figure_dir / f"{figure_name}.pdf"
# png_path = figure_dir / f"{figure_name}.png"

# fig_lexical_trends.savefig(
#     pdf_path,
#     format="pdf",
#     bbox_inches="tight"
# )

# fig_lexical_trends.savefig(
#     png_path,
#     format="png",
#     dpi=600,
#     bbox_inches="tight"
# )

# print("Supplementary Figure S03 exported.")
# print(pdf_path)
# print(png_path)



## 8. Corpus Vocabulary and Frequency Diagnostics

### Objective

Characterize the vocabulary distributions of the three-country corpus before identifying country-specific keywords and diplomatic themes.

### Measures

- Total lexical tokens by country
- Unique vocabulary size
- Term frequency
- Document frequency
- Normalized term frequency per 10,000 lexical tokens

### Methodological considerations

All documents are tokenized using the previously defined lexical tokenization procedure.

No stopwords are removed during this initial diagnostic stage. This allows the prevalence of grammatical and institutional vocabulary to be examined before establishing a transparent preprocessing policy.

Normalized word frequency is descriptive and does not measure the statistical distinctiveness of a term.

Country-specific keywords, thematic salience, and rhetorical framing require additional methods and validation.


In [ ]:

# ============================================================
# STEP 08: CORPUS VOCABULARY DIAGNOSTICS
# ============================================================

from collections import Counter

country_term_counts = {}
country_document_counts = {}
vocabulary_records = []

for code in ["IRN", "USA", "CHN"]:

    subset = df.loc[df["country_code"] == code]

    term_counter = Counter()
    document_counter = Counter()

    for text in subset["text_clean"]:
        tokens = extract_lexical_tokens(text)

        term_counter.update(tokens)
        document_counter.update(set(tokens))

    country_term_counts[code] = term_counter
    country_document_counts[code] = document_counter

    total_tokens = sum(term_counter.values())

    vocabulary_records.append({
        "country_code": code,
        "n_speeches": len(subset),
        "total_tokens": total_tokens,
        "vocabulary_size": len(term_counter)
    })

vocabulary_summary = pd.DataFrame(vocabulary_records)

# Validate consistency with previously computed lexical counts.
for row in vocabulary_summary.itertuples(index=False):

    expected_tokens = lexical_df.loc[
        lexical_df["country_code"] == row.country_code,
        "lexical_token_count"
    ].sum()

    assert row.total_tokens == expected_tokens

print("=" * 70)
print("CORPUS VOCABULARY DIAGNOSTICS")
print("=" * 70)

display(vocabulary_summary)

# Construct the top-term diagnostic table.
top_term_records = []

for code in ["IRN", "USA", "CHN"]:

    total_tokens = sum(country_term_counts[code].values())
    n_documents = int(
        (df["country_code"] == code).sum()
    )

    for rank, (term, frequency) in enumerate(
        country_term_counts[code].most_common(30),
        start=1
    ):
        document_frequency = (
            country_document_counts[code][term]
        )

        top_term_records.append({
            "country_code": code,
            "rank": rank,
            "term": term,
            "term_frequency": frequency,
            "document_frequency": document_frequency,
            "document_prevalence_pct":
                100 * document_frequency / n_documents,
            "frequency_per_10k":
                10000 * frequency / total_tokens
        })

top_terms_df = pd.DataFrame(top_term_records)

assert len(top_terms_df) == 90
assert top_terms_df["document_prevalence_pct"].between(
    0, 100
).all()

print("\nTop 30 lexical terms by country:")

for code in ["IRN", "USA", "CHN"]:
    print(f"\n{COUNTRY_LABELS[code]}")
    display(
        top_terms_df.loc[
            top_terms_df["country_code"] == code
        ].round(2)
    )

print("\nVocabulary diagnostics: PASSED")



## 9. Distinctive Vocabulary: Preliminary Analysis

### Objective

Identify lexical terms that are disproportionately frequent in the speeches of one country relative to the other two countries.

### Analytical strategy

1. Apply a documented English stopword list only to the lexical analysis layer.
2. Preserve potentially meaningful diplomatic vocabulary.
3. Exclude extremely rare terms using corpus-frequency and document-frequency thresholds.
4. Calculate smoothed log2 relative-frequency ratios.
5. Report term frequency, document prevalence, and relative lexical prominence.

### Methodological limitations

These measures are descriptive and do not establish statistical significance.

Country-level aggregation may conceal historical variation. A term may appear distinctive because it was prevalent during a particular geopolitical period.

Proper nouns, country names, and institutional terms may dominate rankings and must be evaluated separately before interpreting thematic priorities.

The complete source texts remain unchanged.


In [ ]:

# ============================================================
# STEP 09: PRELIMINARY DISTINCTIVE VOCABULARY ANALYSIS
# ============================================================

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from collections import Counter
import numpy as np
import pandas as pd

CODES = ["IRN", "USA", "CHN"]

MIN_TOTAL_FREQUENCY = 15
MIN_TOTAL_DOCUMENT_FREQUENCY = 3
SMOOTHING_ALPHA = 0.5

# ------------------------------------------------------------
# 1. Define an explicit lexical filtering policy
# ------------------------------------------------------------

stopwords = set(ENGLISH_STOP_WORDS)

# Preserve substantive terms if included in the stopword list.
protected_terms = {
    "peace", "security", "war", "state", "states",
    "nation", "nations", "international", "national",
    "government", "governments", "united", "system",
    "development", "human", "rights", "justice",
    "economic", "political", "world"
}

stopwords -= protected_terms

all_term_counts = Counter()
all_document_counts = Counter()

for code in CODES:
    all_term_counts.update(country_term_counts[code])
    all_document_counts.update(country_document_counts[code])

candidate_terms = sorted(
    term for term, count in all_term_counts.items()
    if (
        term not in stopwords
        and len(term) >= 3
        and count >= MIN_TOTAL_FREQUENCY
        and all_document_counts[term]
            >= MIN_TOTAL_DOCUMENT_FREQUENCY
    )
)

# ------------------------------------------------------------
# 2. Construct a consistent vocabulary
# ------------------------------------------------------------

vocabulary_size = len(candidate_terms)

filtered_counts = {
    code: Counter({
        term: country_term_counts[code][term]
        for term in candidate_terms
    })
    for code in CODES
}

filtered_totals = {
    code: sum(filtered_counts[code].values())
    for code in CODES
}

assert vocabulary_size > 0
assert all(total > 0 for total in filtered_totals.values())

# ------------------------------------------------------------
# 3. Calculate smoothed log2 frequency ratios
# ------------------------------------------------------------

distinctive_records = []

for code in CODES:

    comparison_codes = [
        other for other in CODES if other != code
    ]

    target_total = filtered_totals[code]

    comparison_total = sum(
        filtered_totals[other]
        for other in comparison_codes
    )

    n_documents = int(
        (df["country_code"] == code).sum()
    )

    for term in candidate_terms:

        target_count = filtered_counts[code][term]

        comparison_count = sum(
            filtered_counts[other][term]
            for other in comparison_codes
        )

        target_rate = (
            (target_count + SMOOTHING_ALPHA)
            / (
                target_total
                + SMOOTHING_ALPHA * vocabulary_size
            )
        )

        comparison_rate = (
            (comparison_count + SMOOTHING_ALPHA)
            / (
                comparison_total
                + SMOOTHING_ALPHA * vocabulary_size
            )
        )

        log2_ratio = np.log2(
            target_rate / comparison_rate
        )

        document_frequency = (
            country_document_counts[code][term]
        )

        distinctive_records.append({
            "country_code": code,
            "term": term,
            "term_frequency": target_count,
            "document_frequency": document_frequency,
            "document_prevalence_pct":
                100 * document_frequency / n_documents,
            "log2_relative_frequency": log2_ratio
        })

distinctive_df = pd.DataFrame(distinctive_records)

# ------------------------------------------------------------
# 4. Inspect prominent terms
# ------------------------------------------------------------

print("=" * 70)
print("DISTINCTIVE VOCABULARY DIAGNOSTICS")
print("=" * 70)

print("Candidate vocabulary size:", vocabulary_size)

for code in CODES:

    subset = distinctive_df.loc[
        (distinctive_df["country_code"] == code)
        & (distinctive_df["term_frequency"] >= 10)
        & (distinctive_df["document_frequency"] >= 3)
    ]

    top_terms = subset.nlargest(
        20, "log2_relative_frequency"
    )

    print(f"\n{COUNTRY_LABELS[code]}: Top distinctive terms")

    display(top_terms.round(3))

print("\nDistinctive vocabulary diagnostics: PASSED")



## 10. Temporal Distribution of Distinctive Terms

### Objective

Assess whether the highest-ranked distinctive terms are broadly distributed across historical periods or concentrated in particular decades.

### Analytical approach

1. Select the top 20 previously identified distinctive terms for each country.
2. Calculate their document-level frequencies.
3. Aggregate frequencies by calendar decade.
4. Normalize term occurrences per 10,000 lexical tokens.
5. Identify the decade contributing the largest proportion of a term's total occurrences.
6. Quantify the number of decades and speeches in which each term appears.

### Methodological limitations

The selected terms originate from an exploratory country-level ranking, which is sensitive to vocabulary selection and smoothing choices.

Decade-level concentration does not establish thematic importance or causal associations with geopolitical events.

The first and last calendar decades contain fewer years than the intervening decades.

The analysis is diagnostic only; no inferential tests or automatic term exclusions are performed.


In [ ]:

# ============================================================
# STEP 10: TEMPORAL CONCENTRATION OF DISTINCTIVE TERMS
# ============================================================

CODES = ["IRN", "USA", "CHN"]
TOP_N = 20

# ------------------------------------------------------------
# 1. Select terms from the existing distinctive ranking
# ------------------------------------------------------------

selected_terms = {}

for code in CODES:

    eligible = distinctive_df.loc[
        (distinctive_df["country_code"] == code)
        & (distinctive_df["term_frequency"] >= 10)
        & (distinctive_df["document_frequency"] >= 3)
    ]

    selected_terms[code] = (
        eligible.nlargest(
            TOP_N, "log2_relative_frequency"
        )["term"].tolist()
    )

    assert len(selected_terms[code]) == TOP_N

# ------------------------------------------------------------
# 2. Build document-level term frequencies
# ------------------------------------------------------------

temporal_term_records = []
document_denominators = []

for row in df.itertuples(index=False):

    tokens = extract_lexical_tokens(row.text_clean)
    counts = Counter(tokens)

    document_denominators.append({
        "country_code": row.country_code,
        "year": row.year,
        "decade": (row.year // 10) * 10,
        "lexical_tokens": len(tokens)
    })

    for term in selected_terms[row.country_code]:

        temporal_term_records.append({
            "country_code": row.country_code,
            "year": row.year,
            "decade": (row.year // 10) * 10,
            "term": term,
            "term_count": counts[term],
            "present": counts[term] > 0
        })

temporal_terms = pd.DataFrame(temporal_term_records)
document_denominators = pd.DataFrame(document_denominators)

assert len(temporal_terms) == len(df) * TOP_N
assert len(document_denominators) == len(df)

# ------------------------------------------------------------
# 3. Calculate decade-specific token denominators
# ------------------------------------------------------------

decade_denominators = (
    document_denominators
    .groupby(["country_code", "decade"])
    .agg(
        decade_tokens=("lexical_tokens", "sum"),
        decade_speeches=("year", "size")
    )
    .reset_index()
)

decade_terms = (
    temporal_terms
    .groupby(["country_code", "term", "decade"])
    .agg(
        term_occurrences=("term_count", "sum"),
        speeches_with_term=("present", "sum")
    )
    .reset_index()
    .merge(
        decade_denominators,
        on=["country_code", "decade"],
        how="left",
        validate="many_to_one"
    )
)

decade_terms["frequency_per_10k"] = (
    10000
    * decade_terms["term_occurrences"]
    / decade_terms["decade_tokens"]
)

# ------------------------------------------------------------
# 4. Quantify historical concentration
# ------------------------------------------------------------

concentration_records = []

for (code, term), group in decade_terms.groupby(
    ["country_code", "term"]
):

    total_occurrences = int(
        group["term_occurrences"].sum()
    )

    total_documents = int(
        group["speeches_with_term"].sum()
    )

    peak_row = group.loc[
        group["term_occurrences"].idxmax()
    ]

    concentration_records.append({
        "country_code": code,
        "term": term,
        "total_occurrences": total_occurrences,
        "speeches_with_term": total_documents,
        "decades_with_term": int(
            group["term_occurrences"].gt(0).sum()
        ),
        "peak_decade": int(peak_row["decade"]),
        "peak_decade_share_pct": (
            100
            * peak_row["term_occurrences"]
            / total_occurrences
            if total_occurrences > 0 else np.nan
        )
    })

term_concentration = pd.DataFrame(
    concentration_records
)

# ------------------------------------------------------------
# 5. Validate against the original term counts
# ------------------------------------------------------------

for row in term_concentration.itertuples(index=False):

    assert (
        row.total_occurrences
        == country_term_counts[row.country_code][row.term]
    )

    assert (
        row.speeches_with_term
        == country_document_counts[row.country_code][row.term]
    )

assert len(term_concentration) == 3 * TOP_N

# ------------------------------------------------------------
# 6. Display diagnostic results
# ------------------------------------------------------------

print("=" * 70)
print("TEMPORAL CONCENTRATION OF DISTINCTIVE VOCABULARY")
print("=" * 70)

for code in CODES:

    print(f"\n{COUNTRY_LABELS[code]}")

    display(
        term_concentration.loc[
            term_concentration["country_code"] == code
        ].sort_values(
            "peak_decade_share_pct",
            ascending=False
        ).round(2)
    )

print("\nTemporal term concentration: PASSED")



## 11. Keyword-in-Context (KWIC) Analysis

### Objective

Examine the contextual use of selected diplomatic terms across Iran, the United States, and China.

### Analytical approach

1. Identify occurrences of predefined exploratory terms.
2. Retrieve surrounding text from the minimally normalized speeches.
3. Preserve document identifiers, country codes, and years.
4. Apply reproducible random sampling when a term has many occurrences.
5. Examine contextual variation without automatically assigning sentiment or political meaning.

### Methodological safeguards

Keyword frequency does not establish thematic importance, sentiment, or diplomatic stance.

Context windows may not contain sufficient information to identify rhetorical framing. Ambiguous cases will require examination of complete sentences or paragraphs.

Sampling is reproducible, and the number of eligible occurrences is reported separately from the number sampled.

No documents or text passages are modified or excluded.


In [ ]:

# ============================================================
# STEP 11: KEYWORD-IN-CONTEXT DIAGNOSTICS
# ============================================================

import re
import numpy as np
import pandas as pd

RANDOM_SEED = 42
CONTEXT_CHARACTERS = 100
MAX_CONTEXTS_PER_COUNTRY_TERM = 5

EXPLORATORY_TERMS = {
    "IRN": ["islamic", "zionist", "security", "peace"],
    "USA": ["democracies", "migration", "security", "peace"],
    "CHN": ["hegemonism", "multipolarity", "security", "peace"]
}

# ------------------------------------------------------------
# 1. Extract occurrences with source identifiers
# ------------------------------------------------------------

kwic_records = []

for row in df.itertuples(index=False):

    terms = EXPLORATORY_TERMS[row.country_code]
    text = row.text_clean

    for term in terms:

        # Match complete terms using Unicode word boundaries.
        pattern = re.compile(
            rf"(?<!\w){re.escape(term)}(?!\w)",
            flags=re.IGNORECASE
        )

        for match in pattern.finditer(text):

            start = max(
                0, match.start() - CONTEXT_CHARACTERS
            )

            end = min(
                len(text),
                match.end() + CONTEXT_CHARACTERS
            )

            kwic_records.append({
                "country_code": row.country_code,
                "year": row.year,
                "archive_path": row.archive_path,
                "term": term,
                "match_start": match.start(),
                "left_context": text[start:match.start()],
                "matched_text": match.group(),
                "right_context": text[match.end():end]
            })

kwic_df = pd.DataFrame(kwic_records)

assert not kwic_df.empty

# ------------------------------------------------------------
# 2. Calculate the complete occurrence inventory
# ------------------------------------------------------------

kwic_inventory = (
    kwic_df
    .groupby(["country_code", "term"])
    .agg(
        n_occurrences=("match_start", "size"),
        n_speeches=("archive_path", "nunique"),
        first_year=("year", "min"),
        last_year=("year", "max")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 3. Reproducible sampling for contextual inspection
# ------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)
sample_indices = []

for _, group in kwic_df.groupby(
    ["country_code", "term"],
    sort=True
):

    n = min(
        len(group),
        MAX_CONTEXTS_PER_COUNTRY_TERM
    )

    chosen = rng.choice(
        group.index.to_numpy(),
        size=n,
        replace=False
    )

    sample_indices.extend(chosen.tolist())

kwic_sample = (
    kwic_df.loc[sample_indices]
    .sort_values(["country_code", "term", "year"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Display results
# ------------------------------------------------------------

print("=" * 70)
print("KEYWORD-IN-CONTEXT DIAGNOSTICS")
print("=" * 70)

print(f"Total eligible occurrences: {len(kwic_df):,}")
print(f"Sampled contexts: {len(kwic_sample):,}")
print(f"Random seed: {RANDOM_SEED}")

print("\nTerm occurrence inventory:")
display(kwic_inventory)

print("\nSampled contexts:")
display(
    kwic_sample[
        [
            "country_code",
            "year",
            "term",
            "left_context",
            "matched_text",
            "right_context"
        ]
    ]
)

print("\nKWIC extraction: PASSED")



## 12. Document-Balanced Contextual Validation

### Objective

Improve contextual inspection by reducing the overrepresentation of speeches containing repeated occurrences of the same keyword.

### Sampling protocol

1. Treat each country-term combination independently.
2. Identify all eligible speeches containing the target term.
3. Randomly select up to ten distinct speeches per country-term combination.
4. Select one keyword occurrence from each sampled speech.
5. Extract an expanded contextual window for qualitative inspection.
6. Preserve speech identifiers and exact character offsets.

### Methodological considerations

This procedure balances selection at the document level but does not ensure equal historical-period representation.

The resulting sample is exploratory and is not intended to estimate population-level framing proportions.

Context excerpts may still be insufficient for identifying negation, agency, rhetorical stance, or references to other actors.

The original texts are preserved without modification.


In [ ]:

# ============================================================
# STEP 12: DOCUMENT-BALANCED CONTEXTUAL SAMPLING
# ============================================================

RANDOM_SEED = 42
MAX_DOCUMENTS_PER_TERM = 10
CONTEXT_WIDTH = 250

rng = np.random.default_rng(RANDOM_SEED)

# ------------------------------------------------------------
# 1. Construct document-level occurrence groups
# ------------------------------------------------------------

balanced_records = []

for (code, term), term_group in kwic_df.groupby(
    ["country_code", "term"],
    sort=True
):

    document_groups = {
        path: group
        for path, group in term_group.groupby("archive_path")
    }

    available_documents = sorted(document_groups)

    n_select = min(
        MAX_DOCUMENTS_PER_TERM,
        len(available_documents)
    )

    selected_documents = rng.choice(
        available_documents,
        size=n_select,
        replace=False
    )

    for archive_path in selected_documents:

        occurrences = document_groups[archive_path]

        chosen_index = rng.choice(
            occurrences.index.to_numpy()
        )

        occurrence = occurrences.loc[chosen_index]

        source = df.loc[
            df["archive_path"] == archive_path
        ].iloc[0]

        text = source["text_clean"]
        position = int(occurrence["match_start"])
        match_length = len(occurrence["matched_text"])

        start = max(0, position - CONTEXT_WIDTH)
        end = min(
            len(text),
            position + match_length + CONTEXT_WIDTH
        )

        context = text[start:end]

        balanced_records.append({
            "country_code": code,
            "year": int(source["year"]),
            "archive_path": archive_path,
            "term": term,
            "match_start": position,
            "context": context,
            "occurrences_in_document": len(occurrences)
        })

balanced_kwic = pd.DataFrame(balanced_records)

# ------------------------------------------------------------
# 2. Validate the sampling design
# ------------------------------------------------------------

assert not balanced_kwic.duplicated(
    ["country_code", "term", "archive_path"]
).any()

assert balanced_kwic.groupby(
    ["country_code", "term"]
).size().le(MAX_DOCUMENTS_PER_TERM).all()

assert balanced_kwic["archive_path"].isin(
    df["archive_path"]
).all()

# Verify that every selected occurrence matches its keyword.
text_lookup = df.set_index("archive_path")["text_clean"]

for row in balanced_kwic.itertuples(index=False):
    source_text = text_lookup.loc[row.archive_path]
    extracted = source_text[
        row.match_start:row.match_start + len(row.term)
    ]
    assert extracted.casefold() == row.term.casefold()

# ------------------------------------------------------------
# 3. Describe selected documents
# ------------------------------------------------------------

sample_summary = (
    balanced_kwic.groupby(["country_code", "term"])
    .agg(
        selected_speeches=("archive_path", "nunique"),
        earliest_year=("year", "min"),
        latest_year=("year", "max"),
        median_occurrences_per_speech=(
            "occurrences_in_document", "median"
        )
    )
    .reset_index()
)

print("=" * 70)
print("DOCUMENT-BALANCED CONTEXTUAL VALIDATION")
print("=" * 70)

print("Selected contexts:", len(balanced_kwic))
print(
    "Distinct source speeches:",
    balanced_kwic["archive_path"].nunique()
)

print("\nSampling summary:")
display(sample_summary)

print("\nSelected examples:")

for code in ["IRN", "USA", "CHN"]:

    print(f"\n{COUNTRY_LABELS[code]}")

    examples = balanced_kwic.loc[
        balanced_kwic["country_code"] == code
    ].sort_values(["term", "year"])

    display(
        examples[
            ["year", "term", "context"]
        ].head(8)
    )

print("\nDocument-balanced sampling: PASSED")
